# Notebook 05 — End-to-end advisor scenario

**ATLAS: Aligned Three-Layer Architecture for Semantics**  
FSI (Financial Services Industry) Semantic Layer Workshop on AWS — Workshop 2

---

Walk the advisor scenario across both UIs.

This notebook traces the full cross-persona flow from signal detection to
conversational follow-up. By the end, you will have seen how the Consumer Banker
and Wealth Advisor workflows connect through the shared backbone — and you will
understand why the audit trail must span both personas.

In [ ]:
# WS2 notebook setup — installs dependencies into THIS kernel.
# uv manages the WS2 virtual environment; this cell installs it
# into the running kernel so imports work without manual setup.
import sys, subprocess, os
from pathlib import Path

# Find the use-case-applications root (3 levels up from phase-1-referral/)
ws2_root = Path(os.getcwd()).parents[1]
venv_python = ws2_root / '.venv' / 'bin' / 'python'

if venv_python.exists() and str(venv_python) != sys.executable:
    # venv exists but we're not running inside it — install into current kernel
    subprocess.check_call(
        [sys.executable, '-m', 'pip', 'install', '--quiet',
         '--disable-pip-version-check',
         'rdflib>=7.0.0', 'pyshacl>=0.25.0', 'SPARQLWrapper>=2.0.0',
         'pydantic>=2.0.0'],
        cwd='/tmp'
    )
elif not venv_python.exists():
    # venv not yet created — run uv sync first
    subprocess.check_call(
        ['uv', 'sync', '--all-groups', '--quiet'],
        cwd=str(ws2_root)
    )
    subprocess.check_call(
        [sys.executable, '-m', 'pip', 'install', '--quiet',
         '--disable-pip-version-check',
         'rdflib>=7.0.0', 'pyshacl>=0.25.0', 'SPARQLWrapper>=2.0.0',
         'pydantic>=2.0.0'],
        cwd='/tmp'
    )

print('WS2 dependencies ready.')


## Key terms for this notebook

| Term | What it is |
|------|------------|
| **Cross-UI flow** | A workflow that begins in one UI (Wholesale) and completes in another (Wealth). The referral starts with the Consumer Banker and lands with the Wealth Advisor. The audit trail must span both. |
| **Routing decision** | The moment the referral-orchestrator sends a referral to the Wealth Advisor's queue. This decision links the two UIs: it references the signal detected in the Wholesale UI and creates a notification in the Wealth UI. |
| **Audit trail** | The PROV-O-attributed chain of events from signal detection through routing to advisor action. Must be queryable across both personas to satisfy compliance requirements. |
| **Notification** | The event that appears in the Wealth UI when a referral is routed. Contains the client URI, signal type, rationale summary, and routing timestamp. |
| **Conversational follow-up** | The Wealth Advisor's ability to ask questions about the referred client using the conversational-context-manager, with the referral context pre-loaded into session memory. |

## The full flow connects two personas through one backbone

The ATLAS advisor scenario is not contained within a single UI. It begins when a
Consumer Banker's wealth-signal-detector identifies a client with a coverage gap
and a qualifying signal — say, a LargeInboundWire or EngagementDecay pattern. The
Consumer Banker reviews the signal in the Wholesale UI, the referral-rationale-drafter
produces a narrative explanation, the banker approves it, and the referral-orchestrator
routes the referral to the appropriate Wealth Advisor. At that point, the workflow
crosses the UI boundary.

The Wealth Advisor receives a notification in the Wealth UI. The notification
contains the client URI, the signal that triggered the referral, and the approved
rationale. The advisor opens the client's profile — rendered through the
CustomerCoverageFragment — and sees behavioral signals, AUM, themes, and engagement
metrics. They can then use the conversational surface to ask follow-up questions:
"What themes are relevant to this client?" or "Show me their engagement history."
The conversational-context-manager resolves these questions using the referral
context that was pre-loaded into session memory when the notification was opened.

The audit trail must span this entire flow. A compliance officer querying the graph
should be able to trace from the original signal detection (who detected it, when,
what evidence) through the rationale (who drafted it, who approved it) through the
routing decision (which orchestrator, which queue, which advisor) to the advisor's
action (when they opened it, what they queried). This is why the routing decision
creates a PROV-O-attributed AuditRecord that references both the Consumer Banker's
signal and the Wealth Advisor's notification. The record links the two UIs in the
graph.

This cross-UI flow is what validates Thesis 2 at the workflow level. It is not
enough for two UIs to render different views of the same data — they must also
participate in shared workflows where actions in one UI produce effects in the
other. The referral flow proves this: the Consumer Banker's action (approve
referral) creates the Wealth Advisor's context (notification + pre-loaded memory).
Same backbone, different personas, connected workflow.

In [ ]:
import sys
import os
import json
import uuid
from datetime import datetime, timezone

# Workshop 1's shared helpers.
sys.path.insert(0, "../../../agentic-semantic-layer/notebooks/shared")

from pathlib import Path

SPEC_DIR = "../../spec/04-aws-agent-registry"

print("Setup complete.")
print("This notebook simulates the cross-UI flow locally.")

In [ ]:
# Build cell 1 — Step 1: Consumer Banker detects signal.
#
# The wealth-signal-detector fires on a client with engagement decay.

signal_event = {
    "step": 1,
    "action": "signal_detected",
    "persona": "atlas-consumer-banker",
    "ui": "Wholesale UI",
    "agent": "wealth-signal-detector",
    "client_uri": "atlas:client-rachel-kim",
    "signal_type": "EngagementDecay",
    "evidence": {
        "baseline_sessions_per_month": 12,
        "recent_90d_sessions": 4,
        "decay_ratio": 0.111,
    },
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "audit_id": str(uuid.uuid4()),
}

print("Step 1: Signal detected in Wholesale UI")
print(json.dumps(signal_event, indent=2))

In [ ]:
# Build cell 2 — Steps 2-3: Draft rationale and route referral.

# Step 2: Rationale drafted
rationale_event = {
    "step": 2,
    "action": "rationale_drafted",
    "persona": "atlas-consumer-banker",
    "agent": "referral-rationale-drafter",
    "client_uri": signal_event["client_uri"],
    "rationale": "Client shows significant engagement decay (decay ratio 0.111). "
                 "Historical baseline of 12 sessions/month dropped to 4 over 90 days. "
                 "Recommend wealth advisor outreach to re-engage.",
    "is_probabilistic": True,
    "requires_human_review": True,
    "approved_by": "banker-jane-doe",
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "audit_id": str(uuid.uuid4()),
    "parent_audit_id": signal_event["audit_id"],
}

# Step 3: Routing decision
routing_event = {
    "step": 3,
    "action": "referral_routed",
    "persona": "atlas-consumer-banker",
    "agent": "referral-orchestrator",
    "client_uri": signal_event["client_uri"],
    "routed_to_persona": "atlas-wealth-advisor",
    "routed_to_advisor": "advisor-michael-ross",
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "audit_id": str(uuid.uuid4()),
    "parent_audit_id": rationale_event["audit_id"],
}

print("Step 2: Rationale drafted and approved")
print(f"  Rationale: {rationale_event['rationale'][:80]}...")
print(f"  Approved by: {rationale_event['approved_by']}")
print()
print("Step 3: Referral routed")
print(f"  Routed to: {routing_event['routed_to_advisor']}")
print(f"  Persona:   {routing_event['routed_to_persona']}")

In [ ]:
# Build cell 3 — Steps 4-6: Wealth Advisor receives and acts.

# Step 4: Notification received in Wealth UI
notification_event = {
    "step": 4,
    "action": "notification_received",
    "persona": "atlas-wealth-advisor",
    "ui": "Wealth UI",
    "client_uri": signal_event["client_uri"],
    "signal_type": signal_event["signal_type"],
    "rationale_summary": rationale_event["rationale"][:100],
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "audit_id": str(uuid.uuid4()),
    "parent_audit_id": routing_event["audit_id"],
}

# Step 5: Advisor opens client profile
profile_event = {
    "step": 5,
    "action": "client_profile_opened",
    "persona": "atlas-wealth-advisor",
    "ui": "Wealth UI",
    "client_uri": signal_event["client_uri"],
    "behavioral_signals": ["EngagementDecay"],
    "themes": ["ESG Transition", "Rate Sensitivity"],
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "audit_id": str(uuid.uuid4()),
    "parent_audit_id": notification_event["audit_id"],
}

# Step 6: Conversational follow-up
conversation_event = {
    "step": 6,
    "action": "conversational_followup",
    "persona": "atlas-wealth-advisor",
    "ui": "Wealth UI",
    "agent": "conversational-context-manager",
    "question": "What is this client's current AUM and engagement trend?",
    "context_from_referral": True,
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "audit_id": str(uuid.uuid4()),
    "parent_audit_id": profile_event["audit_id"],
}

print("Step 4: Notification received in Wealth UI")
print(f"  Signal: {notification_event['signal_type']}")
print()
print("Step 5: Client profile opened")
print(f"  Behavioral signals: {profile_event['behavioral_signals']}")
print(f"  Themes: {profile_event['themes']}")
print()
print("Step 6: Conversational follow-up")
print(f"  Question: {conversation_event['question']}")

In [ ]:
# Build cell 4 — Assemble the full audit trail.

audit_trail = [
    signal_event,
    rationale_event,
    routing_event,
    notification_event,
    profile_event,
    conversation_event,
]

print("Full cross-UI audit trail:")
print("=" * 60)
for event in audit_trail:
    ui = event.get("ui", "—")
    print(f"  Step {event['step']}: [{ui:<12}] {event['action']}")
    print(f"           persona: {event['persona']}")
    print(f"           audit_id: {event['audit_id'][:8]}...")
    if event.get("parent_audit_id"):
        print(f"           parent:   {event['parent_audit_id'][:8]}...")
    print()

# Verify chain integrity
personas_in_trail = {e["persona"] for e in audit_trail}
uis_in_trail = {e.get("ui") for e in audit_trail if e.get("ui")}
print(f"Personas spanned: {sorted(personas_in_trail)}")
print(f"UIs spanned:      {sorted(uis_in_trail)}")

## Verification

Two properties must hold: the audit trail spans both personas (proving the
cross-UI flow is traceable), and the routing decision links to both UIs (proving
the backbone connects the two applications). If either fails, the end-to-end
scenario is incomplete.

In [ ]:
# Verification cell 1 — Audit trail spans both personas.

print("Verifying audit trail spans both personas...")
print()

personas_in_trail = {e["persona"] for e in audit_trail}
expected_personas = {"atlas-consumer-banker", "atlas-wealth-advisor"}

print(f"Personas in audit trail: {sorted(personas_in_trail)}")
print(f"Expected personas:       {sorted(expected_personas)}")
print()

# Verify parent chain is unbroken
audit_ids = {e["audit_id"] for e in audit_trail}
broken_links = []
for event in audit_trail:
    parent = event.get("parent_audit_id")
    if parent and parent not in audit_ids:
        broken_links.append(event["step"])

print(f"Broken parent links: {broken_links if broken_links else 'none'}")
print()

if not expected_personas.issubset(personas_in_trail):
    print("VERIFICATION FAILED: Audit trail does not span both personas.")
    print("The cross-UI flow must include events from both Consumer Banker")
    print("and Wealth Advisor to be compliance-complete.")

assert expected_personas.issubset(personas_in_trail), (
    "Audit trail must span both atlas-consumer-banker and atlas-wealth-advisor. "
    "Cross-UI traceability is required for compliance."
)
assert not broken_links, (
    f"Audit trail has broken parent links at steps: {broken_links}. "
    "Every event must reference its parent for chain integrity."
)

print("[PASS] Audit trail spans both personas with unbroken parent chain.")

In [ ]:
# Verification cell 2 — Routing decision links to both UIs.

print("Verifying routing decision links both UIs...")
print()

# The routing event (step 3) should be traceable from both directions:
# - Forward: routing → notification (Wealth UI)
# - Backward: routing ← rationale ← signal (Wholesale UI)

routing = next(e for e in audit_trail if e["action"] == "referral_routed")
notification = next(e for e in audit_trail if e["action"] == "notification_received")

# Forward link: notification references routing
forward_linked = notification["parent_audit_id"] == routing["audit_id"]
print(f"Forward link (routing → notification): {forward_linked}")

# Backward link: routing references rationale which references signal
rationale = next(e for e in audit_trail if e["action"] == "rationale_drafted")
signal = next(e for e in audit_trail if e["action"] == "signal_detected")

backward_linked = (
    routing["parent_audit_id"] == rationale["audit_id"]
    and rationale["parent_audit_id"] == signal["audit_id"]
)
print(f"Backward link (routing ← rationale ← signal): {backward_linked}")
print()

# Routing spans both UIs
routing_persona = routing["persona"]
notification_persona = notification["persona"]
spans_both = routing_persona != notification_persona
print(f"Routing persona:      {routing_persona}")
print(f"Notification persona: {notification_persona}")
print(f"Spans both personas:  {spans_both}")
print()

if not (forward_linked and backward_linked and spans_both):
    print("VERIFICATION FAILED: Routing decision does not properly link both UIs.")
    print("The routing event must be the bridge between Wholesale and Wealth UI")
    print("audit trails, with parent references in both directions.")

assert forward_linked and backward_linked and spans_both, (
    "Routing decision must link both UIs: forward to notification (Wealth UI) "
    "and backward to signal (Wholesale UI) through parent audit IDs."
)

print("[PASS] Routing decision correctly links both UIs.")
print("The cross-UI workflow is fully traceable.")

## What just changed

You have walked the full advisor scenario across both UIs. The flow starts with
signal detection in the Wholesale UI, passes through rationale drafting and routing,
and completes with notification receipt and conversational follow-up in the Wealth
UI. The audit trail spans both personas with an unbroken parent chain.

This proves that the two UIs are not independent applications — they are connected
through the shared backbone, with the routing decision as the bridge. A compliance
officer can trace any referral from detection to advisor action across both personas.

The final notebook runs the Phase 2 acceptance suite to confirm that all components
— agents, memory, JWT auth, cross-UI flow — work together as specified.